In [14]:
import pandas as pd
from dataclasses import asdict
from collections import Counter

from core.types import LLMCommandingLabelledDataPoint
from core.analysis.utils import exact_match, normalized_edit_distance, action_type_f1

In [3]:
# data/outputs/all-data-gpt5.csv contains the labels
# Let us consider Qwen2.5-1.5B-Instruct
# Create all-data-qwen2.5-1.5b.csv
# then use core.analysis.utils to compute exact_match, normalized_edit_distance, action_type_f1

test_split_ids = pd.read_csv("data/inputs/test-split-ids.csv", header=None)
manually_evaluated_ids = pd.read_csv("data/outputs/selected-data-low-reasoning-gpt5.csv")

manual_ids = manually_evaluated_ids['input_id'].tolist()
test_ids = test_split_ids[0].tolist() + manual_ids

print(f"Number of items to evaluate automatically: {len(test_ids)}")
print(f"Number of items to evaluate manually: {len(manual_ids)}")

Number of items to evaluate automatically: 182
Number of items to evaluate manually: 50


In [11]:
model1_path = "data/outputs/all-data-gpt5.csv"
model1_df = pd.read_csv(model1_path)

model1_entries = [LLMCommandingLabelledDataPoint(**row) for row in model1_df.to_dict(orient="records")]
model1_test = sorted([entry for entry in model1_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

model2_path = "data/outputs/all-data-qwen-1.5b-untrained.csv"
model2_df = pd.read_csv(model2_path)

model2_entries = [LLMCommandingLabelledDataPoint(**row) for row in model2_df.to_dict(orient="records")]
model2_test = sorted([entry for entry in model2_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

In [15]:
labels = []
for teacher in model1_test:
    teacher.infer_label()
    labels.append(teacher.label)

print(Counter(labels))

Counter({<DataPointLabel.FULLY_CORRECT: 'FC'>: 175, <DataPointLabel.CORRECT_BUT_NOT_OPTIMAL: 'CNO'>: 6, <DataPointLabel.CONCEPTUALLY_WRONG: 'CW'>: 1})


In [16]:
labels = []
for student in model2_test:
    student.infer_label()
    labels.append(student.label)

print(Counter(labels))

Counter({<DataPointLabel.FULLY_CORRECT: 'FC'>: 138, <DataPointLabel.SYNTACTICALLY_WRONG: 'SW'>: 13, <DataPointLabel.CORRECT_BUT_NOT_OPTIMAL: 'CNO'>: 12, <DataPointLabel.OPERATIONALLY_WRONG: 'OW'>: 10, <DataPointLabel.CONCEPTUALLY_WRONG: 'CW'>: 9})


In [12]:
outputs_df = []
for teacher, student in zip(model1_test, model2_test):
    assert teacher.input_id == student.input_id

    student.exact_match = exact_match(teacher.game_actions, student.game_actions)
    student.edit_distance = normalized_edit_distance(teacher.game_actions, student.game_actions)

    teacher_actions = [line.split()[0] for line in teacher.game_actions.replace("ASYNC", "").splitlines() if line.split()]
    student_actions = [line.split()[0] for line in student.game_actions.replace("ASYNC", "").splitlines() if line.split()]

    student.action_type_f1 = action_type_f1(teacher_actions, student_actions)

    student.infer_label()

    outputs_df.append({
        "id": teacher.input_id,
        "command": teacher.command,
        "teacher": teacher.game_actions,
        "student": student.game_actions,
        "em": student.exact_match,
        "ed": student.edit_distance,
        "f1": student.action_type_f1,
        "label": student.label,
    })

outputs_df = pd.DataFrame(outputs_df)

In [13]:
outputs_df

,id,command,teacher,student,em,ed,f1,label
0,state-10643-p0-uc2,Ignore the imp and focus on progressing,SPRINT 0.0 636.92\r\nINTERACT,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,False,0.833333,0.000000,DataPointLabel.FULLY_CORRECT
1,state-10643-p1-uc0,Hit that switch on the wall,SPRINT 0.0 636.92\r\nINTERACT,FIRE_SHOTS 5,False,0.925926,0.000000,DataPointLabel.FULLY_CORRECT
2,state-11021-p0-uc1,Center aim on that imp and fire,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 2.0,False,0.166667,1.000000,DataPointLabel.CORRECT_BUT_NOT_OPTIMAL
3,state-11021-p1-uc2,Focus on the imp after these guys,ROTATE_TO_TARGET MONSTER_1\r\nFIRE_SHOTS 1\r\n...,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,False,0.690909,0.285714,DataPointLabel.FULLY_CORRECT
4,state-11066-p1-uc0,Line up on that imp and fire,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 0.5,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,False,0.190476,1.000000,DataPointLabel.OPERATIONALLY_WRONG
...,...,...,...,...,...,...,...,...
177,state-8567-p0-uc0,Switch to the shotgun and blast the front guy,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_2\r...,ASYNC SELECT Shotgun\r\nMOVE 0 559.43\r\nFIRE_...,False,0.711538,0.333333,DataPointLabel.FULLY_CORRECT
178,state-8567-p0-uc2,Back up while firing at them,SELECT Shotgun\r\nASYNC FIRE 2.0\r\nMOVE 0 -200,ASYNC ROTATE_TO_TARGET MONSTER_2\r\nMOVE 5 0\r...,False,0.769231,0.666667,DataPointLabel.FULLY_CORRECT
179,state-973-p2-uc0,Shoot that imp ahead,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,ASYNC FIRE 1.0,False,0.722222,0.666667,DataPointLabel.FULLY_CORRECT
180,state-991-p2-uc0,Turn toward the closest enemy and attack,FAIL No usable weapon available to attack the ...,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,False,0.921569,0.000000,DataPointLabel.FULLY_CORRECT


In [145]:
rows = []
for entry in model2_test:
    rows.append(asdict(entry))

df = pd.DataFrame(rows)

In [146]:
df

,input_id,game_state,command,command_intent,command_explicitness,command_atomicity,command_contextuality,game_actions,latency,reason_if_failed,...,action_full_correct,action_unnecessary,action_imprecise_sequentiality,action_imprecise_parameters,action_harming_sequentiality,action_harming_parameters,action_missing,action_harming,action_wrong_syntax,label
0,state-10643-p0-uc2,AIMED_AT:\r\n type: Wall\r\n distance: 636.9...,Ignore the imp and focus on progressing,Prioritize reaching and using the interactable...,0.60,0.40,0.85,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.172649,NaN,...,0,0,0,0,0,0,0,0,0,NaN
1,state-10643-p1-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 636.9...,Hit that switch on the wall,Reach the interactable wall and activate the s...,0.85,0.40,0.80,FIRE_SHOTS 5,0.127707,NaN,...,0,0,0,0,0,0,0,0,0,NaN
2,state-11021-p0-uc1,AIMED_AT:\r\n type: Monster\r\n distance: 68...,Center aim on that imp and fire,Adjust aim to the visible imp and shoot it once,0.85,0.50,0.85,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 2.0,0.206606,NaN,...,1,0,1,0,0,0,0,0,0,NaN
3,state-11021-p1-uc2,AIMED_AT:\r\n type: Monster\r\n distance: 68...,Focus on the imp after these guys,"Prioritize shotgun guys first, then eliminate ...",0.60,0.30,0.85,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.162326,NaN,...,0,0,0,0,0,0,0,0,0,NaN
4,state-11066-p1-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 319.0...,Line up on that imp and fire,Aim directly at the visible DoomImp and shoot it,0.90,0.45,0.90,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.194801,NaN,...,1,0,0,0,1,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,state-8567-p0-uc0,AIMED_AT:\r\n type: Monster\r\n distance: 55...,Switch to the shotgun and blast the front guy,Equip the shotgun and shoot the monster I’m ai...,0.90,0.60,0.85,ASYNC SELECT Shotgun\r\nMOVE 0 559.43\r\nFIRE_...,0.241976,NaN,...,0,0,0,0,0,0,0,0,0,NaN
178,state-8567-p0-uc2,AIMED_AT:\r\n type: Monster\r\n distance: 55...,Back up while firing at them,Create distance from the group of monsters whi...,0.70,0.30,0.90,ASYNC ROTATE_TO_TARGET MONSTER_2\r\nMOVE 5 0\r...,0.217743,NaN,...,0,0,0,0,0,0,0,0,0,NaN
179,state-973-p2-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 100.3...,Shoot that imp ahead,Damage or kill the visible DoomImp enemy,0.80,0.60,0.85,ASYNC FIRE 1.0,0.123793,NaN,...,0,0,0,0,0,0,0,0,0,NaN
180,state-991-p2-uc0,AIMED_AT:\r\n type: Ceiling\r\n distance: 60...,Turn toward the closest enemy and attack,Engage the nearest visible monster in combat i...,0.70,0.40,0.85,ASYNC ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.193130,NaN,...,0,0,0,0,0,0,0,0,0,NaN


In [17]:
model1_path = "data/outputs/all-data-gpt5.csv"
model1_df = pd.read_csv(model1_path)

model1_entries = [LLMCommandingLabelledDataPoint(**row) for row in model1_df.to_dict(orient="records")]
model1_test = sorted([entry for entry in model1_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

model2_path = "data/outputs/all-data-qwen-1.5b-trained.csv"
model2_df = pd.read_csv(model2_path)

model2_entries = [LLMCommandingLabelledDataPoint(**row) for row in model2_df.to_dict(orient="records")]
model2_test = sorted([entry for entry in model2_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

In [18]:
outputs_df = []
for teacher, student in zip(model1_test, model2_test):
    assert teacher.input_id == student.input_id

    student.exact_match = exact_match(teacher.game_actions, student.game_actions)
    student.edit_distance = normalized_edit_distance(teacher.game_actions, student.game_actions)

    teacher_actions = [line.split()[0] for line in teacher.game_actions.replace("ASYNC", "").splitlines() if line.split()]
    student_actions = [line.split()[0] for line in student.game_actions.replace("ASYNC", "").splitlines() if line.split()]

    student.action_type_f1 = action_type_f1(teacher_actions, student_actions)

    outputs_df.append({
        "id": teacher.input_id,
        "command": teacher.command,
        "teacher": teacher.game_actions,
        "student": student.game_actions,
        "em": student.exact_match,
        "ed": student.edit_distance,
        "f1": student.action_type_f1,
    })

outputs_df = pd.DataFrame(outputs_df)

In [19]:
labels = []
for student in model2_test:
    student.infer_label()
    labels.append(student.label)

print(Counter(labels))

Counter({<DataPointLabel.FULLY_CORRECT: 'FC'>: 180, <DataPointLabel.CORRECT_BUT_NOT_OPTIMAL: 'CNO'>: 2})


In [150]:
outputs_df

,id,command,teacher,student,em,ed,f1
0,state-10643-p0-uc2,Ignore the imp and focus on progressing,SPRINT 0.0 636.92\r\nINTERACT,ROTATE_TO_TARGET MONSTER_0\r\nMOVE_TO_TARGET M...,False,0.865385,0.000000
1,state-10643-p1-uc0,Hit that switch on the wall,SPRINT 0.0 636.92\r\nINTERACT,SPRINT 0.0 636.92\r\nINTERACT,True,0.000000,1.000000
2,state-11021-p0-uc1,Center aim on that imp and fire,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,True,0.000000,1.000000
3,state-11021-p1-uc2,Focus on the imp after these guys,ROTATE_TO_TARGET MONSTER_1\r\nFIRE_SHOTS 1\r\n...,ROTATE_TO_TARGET MONSTER_0\r\nFIRE_SHOTS 2,False,0.645455,0.571429
4,state-11066-p1-uc0,Line up on that imp and fire,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 0.5,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,False,0.055556,1.000000
...,...,...,...,...,...,...,...
177,state-8567-p0-uc0,Switch to the shotgun and blast the front guy,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_2\r...,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_2\r...,False,0.142857,0.666667
178,state-8567-p0-uc2,Back up while firing at them,SELECT Shotgun\r\nASYNC FIRE 2.0\r\nMOVE 0 -200,SELECT Shotgun\r\nASYNC FIRE 2.0\r\nMOVE 0 -150,False,0.046512,1.000000
179,state-973-p2-uc0,Shoot that imp ahead,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,True,0.000000,1.000000
180,state-991-p2-uc0,Turn toward the closest enemy and attack,FAIL No usable weapon available to attack the ...,ROTATE_TO_TARGET MONSTER_1\r\nMOVE_TO_TARGET M...,False,0.933333,0.000000


In [151]:
rows = []
for entry in model2_test:
    rows.append(asdict(entry))

df = pd.DataFrame(rows)

In [152]:
df

,input_id,game_state,command,command_intent,command_explicitness,command_atomicity,command_contextuality,game_actions,latency,reason_if_failed,...,action_full_correct,action_unnecessary,action_imprecise_sequentiality,action_imprecise_parameters,action_harming_sequentiality,action_harming_parameters,action_missing,action_harming,action_wrong_syntax,label
0,state-10643-p0-uc2,AIMED_AT:\r\n type: Wall\r\n distance: 636.9...,Ignore the imp and focus on progressing,Prioritize reaching and using the interactable...,0.60,0.40,0.85,ROTATE_TO_TARGET MONSTER_0\r\nMOVE_TO_TARGET M...,0.167190,NaN,...,0,0,0,0,0,0,0,0,0,NaN
1,state-10643-p1-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 636.9...,Hit that switch on the wall,Reach the interactable wall and activate the s...,0.85,0.40,0.80,SPRINT 0.0 636.92\r\nINTERACT,0.202592,NaN,...,0,0,0,0,0,0,0,0,0,NaN
2,state-11021-p0-uc1,AIMED_AT:\r\n type: Monster\r\n distance: 68...,Center aim on that imp and fire,Adjust aim to the visible imp and shoot it once,0.85,0.50,0.85,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.187625,NaN,...,2,0,0,0,0,0,0,0,0,NaN
3,state-11021-p1-uc2,AIMED_AT:\r\n type: Monster\r\n distance: 68...,Focus on the imp after these guys,"Prioritize shotgun guys first, then eliminate ...",0.60,0.30,0.85,ROTATE_TO_TARGET MONSTER_0\r\nFIRE_SHOTS 2,0.159620,NaN,...,0,0,0,0,0,0,0,0,0,NaN
4,state-11066-p1-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 319.0...,Line up on that imp and fire,Aim directly at the visible DoomImp and shoot it,0.90,0.45,0.90,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.182224,NaN,...,2,0,0,0,0,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,state-8567-p0-uc0,AIMED_AT:\r\n type: Monster\r\n distance: 55...,Switch to the shotgun and blast the front guy,Equip the shotgun and shoot the monster I’m ai...,0.90,0.60,0.85,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_2\r...,0.205678,NaN,...,0,0,0,0,0,0,0,0,0,NaN
178,state-8567-p0-uc2,AIMED_AT:\r\n type: Monster\r\n distance: 55...,Back up while firing at them,Create distance from the group of monsters whi...,0.70,0.30,0.90,SELECT Shotgun\r\nASYNC FIRE 2.0\r\nMOVE 0 -150,0.190103,NaN,...,0,0,0,0,0,0,0,0,0,NaN
179,state-973-p2-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 100.3...,Shoot that imp ahead,Damage or kill the visible DoomImp enemy,0.80,0.60,0.85,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.168639,NaN,...,0,0,0,0,0,0,0,0,0,NaN
180,state-991-p2-uc0,AIMED_AT:\r\n type: Ceiling\r\n distance: 60...,Turn toward the closest enemy and attack,Engage the nearest visible monster in combat i...,0.70,0.40,0.85,ROTATE_TO_TARGET MONSTER_1\r\nMOVE_TO_TARGET M...,0.268357,NaN,...,0,0,0,0,0,0,0,0,0,NaN


In [115]:
model1_path = "data/outputs/all-data-gpt5.csv"
model1_df = pd.read_csv(model1_path)

model1_entries = [LLMCommandingLabelledDataPoint(**row) for row in model1_df.to_dict(orient="records")]
model1_test = sorted([entry for entry in model1_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

model2_path = "data/outputs/all-data-qwen-0.5b-trained.csv"
model2_df = pd.read_csv(model2_path)

model2_entries = [LLMCommandingLabelledDataPoint(**row) for row in model2_df.to_dict(orient="records")]
model2_test = sorted([entry for entry in model2_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

In [116]:
for teacher, student in zip(model1_test, model2_test):
    assert teacher.input_id == student.input_id

    student.exact_match = exact_match(teacher.game_actions, student.game_actions)
    student.edit_distance = normalized_edit_distance(teacher.game_actions, student.game_actions)

    teacher_actions = [line.split()[0] for line in teacher.game_actions.replace("ASYNC", "").splitlines() if line.split()]
    student_actions = [line.split()[0] for line in student.game_actions.replace("ASYNC", "").splitlines() if line.split()]

    student.action_type_f1 = action_type_f1(teacher_actions, student_actions)

In [117]:
rows = []
for entry in model2_test:
    rows.append(asdict(entry))

df = pd.DataFrame(rows)

In [118]:
df

,input_id,game_state,command,command_intent,command_explicitness,command_atomicity,command_contextuality,game_actions,latency,reason_if_failed,...,action_full_correct,action_unnecessary,action_imprecise_sequentiality,action_imprecise_parameters,action_harming_sequentiality,action_harming_parameters,action_missing,action_harming,action_wrong_syntax,label
0,state-10643-p0-uc2,AIMED_AT:\r\n type: Wall\r\n distance: 636.9...,Ignore the imp and focus on progressing,Prioritize reaching and using the interactable...,0.60,0.40,0.85,ROTATE_TO_TARGET MONSTER_0\r\nFIRE_SHOTS 4,0.119644,NaN,...,0,0,0,0,0,0,0,0,0,NaN
1,state-10643-p1-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 636.9...,Hit that switch on the wall,Reach the interactable wall and activate the s...,0.85,0.40,0.80,SPRINT 0.0 636.92\r\nINTERACT,0.160179,NaN,...,0,0,0,0,0,0,0,0,0,NaN
2,state-11021-p0-uc1,AIMED_AT:\r\n type: Monster\r\n distance: 68...,Center aim on that imp and fire,Adjust aim to the visible imp and shoot it once,0.85,0.50,0.85,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.132854,NaN,...,0,0,0,0,0,0,0,0,0,NaN
3,state-11021-p1-uc2,AIMED_AT:\r\n type: Monster\r\n distance: 68...,Focus on the imp after these guys,"Prioritize shotgun guys first, then eliminate ...",0.60,0.30,0.85,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_0\r...,0.201370,NaN,...,0,0,0,0,0,0,0,0,0,NaN
4,state-11066-p1-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 319.0...,Line up on that imp and fire,Aim directly at the visible DoomImp and shoot it,0.90,0.45,0.90,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.135052,NaN,...,0,0,0,0,0,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,state-8567-p0-uc0,AIMED_AT:\r\n type: Monster\r\n distance: 55...,Switch to the shotgun and blast the front guy,Equip the shotgun and shoot the monster I’m ai...,0.90,0.60,0.85,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_2\r...,0.143148,NaN,...,0,0,0,0,0,0,0,0,0,NaN
178,state-8567-p0-uc2,AIMED_AT:\r\n type: Monster\r\n distance: 55...,Back up while firing at them,Create distance from the group of monsters whi...,0.70,0.30,0.90,SELECT Shotgun\r\nASYNC FIRE 2.0\r\nMOVE 0 -150,0.131908,NaN,...,0,0,0,0,0,0,0,0,0,NaN
179,state-973-p2-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 100.3...,Shoot that imp ahead,Damage or kill the visible DoomImp enemy,0.80,0.60,0.85,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,0.140469,NaN,...,0,0,0,0,0,0,0,0,0,NaN
180,state-991-p2-uc0,AIMED_AT:\r\n type: Ceiling\r\n distance: 60...,Turn toward the closest enemy and attack,Engage the nearest visible monster in combat i...,0.70,0.40,0.85,ROTATE_TO_TARGET MONSTER_1\r\nFIRE 1.0,0.125492,NaN,...,0,0,0,0,0,0,0,0,0,NaN
